In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import xgboost as xgb
from PIL import Image
import re


In [ ]:
def download_images(image_links, download_dir):
    if not os.path.exists(download_dir):
        os.makedirs(download_dir)


In [ ]:
DATASET_FOLDER = r'C:\Users\anish\OneDrive\Desktop\New folder\68e8d1d70b66d_student_resource\student_resource\dataset'
train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))


In [ ]:
download_images(train['image_link'], '../train/images')
download_images(test['image_link'], '../test/images')

IMAGES_FOLDER_TRAIN = '../train/images'
IMAGES_FOLDER_TEST = '../test/images'



In [ ]:
def url_to_local_path(url, image_dir):
    try:
        filename = url.split("/")[-1]
        local_path = os.path.join(image_dir, filename)
        return local_path if os.path.exists(local_path) else None
    except:
        return None

train['image_path'] = train['image_link'].apply(lambda x: url_to_local_path(x, IMAGES_FOLDER_TRAIN))
test['image_path'] = test['image_link'].apply(lambda x: url_to_local_path(x, IMAGES_FOLDER_TEST))



In [ ]:
train_df = train.copy()
test_df = test.copy()



In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df["clean_text"] = train_df["catalogue"].apply(clean_text)
test_df["clean_text"] = test_df["catalogue"].apply(clean_text)



In [ ]:
text_model = SentenceTransformer("all-MiniLM-L6-v2")
train_text_emb = text_model.encode(train_df["clean_text"].tolist(), show_progress_bar=True, batch_size=64)
test_text_emb = text_model.encode(test_df["clean_text"].tolist(), show_progress_bar=True, batch_size=64)



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")



In [ ]:
def get_image_embeddings(df, image_folder):
    embeddings = []
    df_with_images = df[df['image_path'].notnull()]
    for local_img_path in tqdm(df_with_images["image_path"], desc=f"Encoding {image_folder} images"):
        try:
            image = Image.open(local_img_path).convert("RGB")
            inputs = clip_processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                emb = clip_model.get_image_features(**inputs)
            embeddings.append(emb.cpu().numpy().flatten())
        except:
            embeddings.append(np.zeros(512))
    temp_embeddings_dict = {path: emb for path, emb in zip(df_with_images["image_path"], embeddings)}
    final_embeddings = []
    for path in df["image_path"]:
        if path in temp_embeddings_dict:
            final_embeddings.append(temp_embeddings_dict[path])
        else:
            final_embeddings.append(np.zeros(512))
    return np.array(final_embeddings)



In [ ]:
train_img_emb = get_image_embeddings(train_df, IMAGES_FOLDER_TRAIN)
test_img_emb = get_image_embeddings(test_df, IMAGES_FOLDER_TEST)

train_features = np.concatenate([train_text_emb, train_img_emb], axis=1)
test_features = np.concatenate([test_text_emb, test_img_emb], axis=1)



In [ ]:
X = train_features
y = train_df["price"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="gpu_hist" if torch.cuda.is_available() else "hist"
)
xgb_model.fit(X_train, y_train)
y_pred_val = xgb_model.predict(X_val)



In [ ]:
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return 100 * np.mean(diff)



In [ ]:
val_smape = smape(y_val, y_pred_val)
print(f"Validation SMAPE: {val_smape:.2f}%")

test_preds = xgb_model.predict(test_features)
test_df["predicted_price"] = test_preds
test_df[["sample_id", "predicted_price"]].to_csv("submission.csv", index=False)
print("Predictions saved to submission.csv")


